In [ ]:
!wget https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
!pip install transformers
!pip install tokenizers
!pip install accelerate

--2025-09-13 10:21:04--  https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26129701 (25M) [text/plain]
Saving to: ‘combined_dataset.csv’

combined_dataset.cs 100%[===================>]  24.92M  --.-KB/s    in 0.1s    

2025-09-13 10:21:05 (224 MB/s) - ‘combined_dataset.csv’ saved [26129701/26129701]



In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from collections import Counter
import time
from datetime import datetime
import os

In [ ]:
dataset = pd.read_csv("combined_dataset.csv")

In [ ]:
class MultiTaskTrainer(Trainer):
    """Custom Trainer for multi-task learning with proper evaluation"""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Custom loss computation for multi-task learning
        """
        outputs = model(**inputs)
        loss = outputs['loss']

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """
        Custom prediction step that returns proper format for evaluation
        """
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs['loss']

            if prediction_loss_only:
                return (loss.detach(), None, None)

            # Return logits as a single concatenated tensor to avoid padding issues
            # We'll separate them later in compute_metrics
            batch_size, seq_len = outputs['ner_logits'].shape[:2]

            # Create a combined prediction tensor with task indicators
            # Format: [batch_size, seq_len, total_features]
            # where total_features = ner_logits + pos_logits + lemma_logits + task_labels
            ner_logits = outputs['ner_logits']  # [batch, seq, ner_labels]
            pos_logits = outputs['pos_logits']  # [batch, seq, pos_labels]
            lemma_logits = outputs['lemma_logits']  # [batch, seq, lemma_labels]

            # Get predictions (argmax)
            ner_preds = torch.argmax(ner_logits, dim=-1)  # [batch, seq]
            pos_preds = torch.argmax(pos_logits, dim=-1)  # [batch, seq]
            lemma_preds = torch.argmax(lemma_logits, dim=-1)  # [batch, seq]

        return (loss.detach(),
                (ner_preds, pos_preds, lemma_preds),
                (inputs['ner_labels'], inputs['pos_labels'], inputs['lemma_labels']))

    # def evaluate(self, dataset=None):
    #     """Evaluate the multi-task model"""
    #     if self.trainer is None:
    #         print("❌ Trainer not available.")
    #         return None

    #     if dataset is None:
    #         dataset = self.test_dataset

    #     if dataset is None:
    #         print("❌ No dataset provided and no test dataset available.")
    #         return None

    #     print("\n🔍 EVALUATING MULTI-TASK BERT MODEL")
    #     print("=" * 50)

    #     try:
    #         eval_dataloader = self.trainer.get_eval_dataloader(dataset)

    #         all_ner_preds, all_pos_preds, all_lemma_preds = [], [], []
    #         all_ner_labels, all_pos_labels, all_lemma_labels = [], [], []

    #         self.model.eval()
    #         for batch in eval_dataloader:
    #             batch = self.trainer._prepare_inputs(batch)
    #             with torch.no_grad():
    #                 outputs = self.model(**batch)

    #                 # Get predictions
    #                 ner_preds = torch.argmax(outputs['ner_logits'], dim=-1).cpu().numpy()
    #                 pos_preds = torch.argmax(outputs['pos_logits'], dim=-1).cpu().numpy()
    #                 lemma_preds = torch.argmax(outputs['lemma_logits'], dim=-1).cpu().numpy()

    #                 # Get labels
    #                 ner_labels = batch['ner_labels'].cpu().numpy()
    #                 pos_labels = batch['pos_labels'].cpu().numpy()
    #                 lemma_labels = batch['lemma_labels'].cpu().numpy()

    #                 all_ner_preds.append(ner_preds)
    #                 all_pos_preds.append(pos_preds)
    #                 all_lemma_preds.append(lemma_preds)
    #                 all_ner_labels.append(ner_labels)
    #                 all_pos_labels.append(pos_labels)
    #                 all_lemma_labels.append(lemma_labels)

    #         # Concatenate all batches
    #         all_ner_preds = np.concatenate(all_ner_preds, axis=0)
    #         all_pos_preds = np.concatenate(all_pos_preds, axis=0)
    #         all_lemma_preds = np.concatenate(all_lemma_preds, axis=0)
    #         all_ner_labels = np.concatenate(all_ner_labels, axis=0)
    #         all_pos_labels = np.concatenate(all_pos_labels, axis=0)
    #         all_lemma_labels = np.concatenate(all_lemma_labels, axis=0)

    #         # Stack into single 3D arrays
    #         predictions = np.stack([all_ner_preds, all_pos_preds, all_lemma_preds], axis=-1)
    #         labels = np.stack([all_ner_labels, all_pos_labels, all_lemma_labels], axis=-1)

    #         # Create EvalPrediction and compute metrics
    #         from transformers.trainer_utils import EvalPrediction
    #         eval_pred = EvalPrediction(predictions=predictions, label_ids=labels)
    #         metrics = self.compute_metrics(eval_pred)

    #         # Return metrics in the same format as Trainer
    #         eval_result = {f"eval_{k}": v for k, v in metrics.items()}

    #         # Print results
    #         print("📊 Multi-Task Evaluation Results:")
    #         print(f"\n🎯 NER Performance:")
    #         print(f"   • Accuracy: {eval_result.get('eval_ner_accuracy', 0):.4f}")
    #         print(f"   • F1 Score: {eval_result.get('eval_ner_f1', 0):.4f}")
    #         print(f"   • Precision: {eval_result.get('eval_ner_precision', 0):.4f}")
    #         print(f"   • Recall: {eval_result.get('eval_ner_recall', 0):.4f}")

    #         print(f"\n🏷️ POS Performance:")
    #         print(f"   • Accuracy: {eval_result.get('eval_pos_accuracy', 0):.4f}")
    #         print(f"   • F1 Score: {eval_result.get('eval_pos_f1', 0):.4f}")
    #         print(f"   • Precision: {eval_result.get('eval_pos_precision', 0):.4f}")
    #         print(f"   • Recall: {eval_result.get('eval_pos_recall', 0):.4f}")

    #         print(f"\n📝 Lemma Performance:")
    #         print(f"   • Accuracy: {eval_result.get('eval_lemma_accuracy', 0):.4f}")
    #         print(f"   • F1 Score: {eval_result.get('eval_lemma_f1', 0):.4f}")
    #         print(f"   • Precision: {eval_result.get('eval_lemma_precision', 0):.4f}")
    #         print(f"   • Recall: {eval_result.get('eval_lemma_recall', 0):.4f}")

    #         print(f"\n🎯 Overall Performance:")
    #         print(f"   • Average F1: {eval_result.get('eval_average_f1', 0):.4f}")

    #         return eval_result

    #     except Exception as e:
    #         print(f"❌ Evaluation failed: {e}")
    #         return None


class MultiTaskBERTModel(nn.Module):
    """Custom BERT model with multiple classification heads for NER, POS, and Lemma"""

    def __init__(self, model_name, num_ner_labels, num_pos_labels, num_lemma_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)

        # Multiple classification heads
        self.ner_classifier = nn.Linear(self.bert.config.hidden_size, num_ner_labels)
        self.pos_classifier = nn.Linear(self.bert.config.hidden_size, num_pos_labels)
        self.lemma_classifier = nn.Linear(self.bert.config.hidden_size, num_lemma_labels)

        # Loss weights for multi-task learning
        self.ner_weight = 1.0
        self.pos_weight = 1.0
        self.lemma_weight = 1.0

    def forward(self, input_ids, attention_mask=None, ner_labels=None, pos_labels=None, lemma_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        # Get predictions from all heads
        ner_logits = self.ner_classifier(sequence_output)
        pos_logits = self.pos_classifier(sequence_output)
        lemma_logits = self.lemma_classifier(sequence_output)

        total_loss = None
        if ner_labels is not None and pos_labels is not None and lemma_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            # Calculate individual losses
            ner_loss = loss_fct(ner_logits.view(-1, self.ner_classifier.out_features), ner_labels.view(-1))
            pos_loss = loss_fct(pos_logits.view(-1, self.pos_classifier.out_features), pos_labels.view(-1))
            lemma_loss = loss_fct(lemma_logits.view(-1, self.lemma_classifier.out_features), lemma_labels.view(-1))

            # Combined weighted loss
            total_loss = (self.ner_weight * ner_loss +
                         self.pos_weight * pos_loss +
                         self.lemma_weight * lemma_loss)

        return {
            'loss': total_loss,
            'ner_logits': ner_logits,
            'pos_logits': pos_logits,
            'lemma_logits': lemma_logits,
            'ner_loss': ner_loss if ner_labels is not None else None,
            'pos_loss': pos_loss if pos_labels is not None else None,
            'lemma_loss': lemma_loss if lemma_labels is not None else None,
        }


class MultiTaskBERT:
    def __init__(self, dataset, model_name="bert-base-multilingual-cased"):
        """
        Initialize Multi-Task BERT model for NER, POS tagging, and Lemmatization

        Args:
            dataset: pandas DataFrame with columns ['SENTENCE #', 'WORD', 'NER_TAG', 'POS_TAG', 'LEMMA']
            model_name: pretrained BERT model name
        """
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dataset = dataset
        self.max_length = 128

        # Get labels for all tasks
        self.ner_label2id, self.ner_id2label = self.get_labels('NER_TAG')
        self.pos_label2id, self.pos_id2label = self.get_labels('POS_TAG')
        self.lemma_label2id, self.lemma_id2label = self.get_labels('LEMMA')

        self.num_ner_labels = len(self.ner_label2id)
        self.num_pos_labels = len(self.pos_label2id)
        self.num_lemma_labels = len(self.lemma_label2id)

        self.sentences = self.get_sentences()

        # Initialize tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        # Initialize custom multi-task model
        self.model = MultiTaskBERTModel(
            model_name=self.model_name,
            num_ner_labels=self.num_ner_labels,
            num_pos_labels=self.num_pos_labels,
            num_lemma_labels=self.num_lemma_labels
        ).to(self.device)

        # Custom data collator for multi-task learning
        self.data_collator = self.create_multi_task_data_collator()

        # Datasets will be created when needed
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.trainer = None

        print(f"🤖 Multi-Task BERT Model Initialized:")
        print(f"   • Model: {self.model_name}")
        print(f"   • Device: {self.device}")
        print(f"   • NER labels: {self.num_ner_labels} ({list(self.ner_label2id.keys())})")
        print(f"   • POS labels: {self.num_pos_labels} ({list(self.pos_label2id.keys())})")
        print(f"   • Lemma labels: {self.num_lemma_labels}")

    def get_labels(self, column_name):
        """Extract unique labels and create mapping dictionaries"""

        labels = sorted(list(set(self.dataset[column_name].values)))
        label2id = {label: i for i, label in enumerate(labels)}
        id2label = {v: k for k, v in label2id.items()}
        return label2id, id2label

    def get_sentences(self):
        """Convert dataset to list of (word, ner_tag, pos_tag, lemma) tuples grouped by sentence"""
        def to_tuples(group):
            return list(zip(
                group["WORD"].values,
                group["NER_TAG"].values,
                group["POS_TAG"].values,
                group["LEMMA"].values
            ))

        sentences = self.dataset.groupby("SENTENCE #").apply(
            to_tuples, include_groups=False
        ).tolist()
        return sentences

    def create_multi_task_data_collator(self):
        """Create custom data collator for multi-task learning"""
        def collate_fn(features):
            batch = {}

            # Standard tokenizer collation
            input_ids = [f['input_ids'] for f in features]
            attention_mask = [f['attention_mask'] for f in features]
            ner_labels = [f['ner_labels'] for f in features]
            pos_labels = [f['pos_labels'] for f in features]
            lemma_labels = [f['lemma_labels'] for f in features]

            # Pad sequences
            max_len = max(len(ids) for ids in input_ids)

            batch['input_ids'] = torch.tensor([
                ids + [self.tokenizer.pad_token_id] * (max_len - len(ids))
                for ids in input_ids
            ])

            batch['attention_mask'] = torch.tensor([
                mask + [0] * (max_len - len(mask))
                for mask in attention_mask
            ])

            batch['ner_labels'] = torch.tensor([
                labels + [-100] * (max_len - len(labels))
                for labels in ner_labels
            ])

            batch['pos_labels'] = torch.tensor([
                labels + [-100] * (max_len - len(labels))
                for labels in pos_labels
            ])

            batch['lemma_labels'] = torch.tensor([
                labels + [-100] * (max_len - len(labels))
                for labels in lemma_labels
            ])

            return batch

        return collate_fn

    def align_labels_with_tokens(self, words, ner_labels, pos_labels, lemma_labels):
        """
        Align all labels with BERT subword tokens

        Args:
            words: list of words
            ner_labels: list of NER labels (as indices)
            pos_labels: list of POS labels (as indices)
            lemma_labels: list of Lemma labels (as indices)

        Returns:
            dict with input_ids, attention_mask, and all aligned labels
        """
        tokenized_inputs = self.tokenizer(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            is_split_into_words=True,
            return_offsets_mapping=True
        )

        aligned_ner_labels = []
        aligned_pos_labels = []
        aligned_lemma_labels = []

        word_ids = tokenized_inputs.word_ids()
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], [PAD])
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)
                aligned_lemma_labels.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the original labels
                aligned_ner_labels.append(ner_labels[word_idx])
                aligned_pos_labels.append(pos_labels[word_idx])
                aligned_lemma_labels.append(lemma_labels[word_idx])
            else:
                # Subsequent subwords of the same word get -100 (ignored in loss)
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)
                aligned_lemma_labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': tokenized_inputs['input_ids'],
            'attention_mask': tokenized_inputs['attention_mask'],
            'ner_labels': aligned_ner_labels,
            'pos_labels': aligned_pos_labels,
            'lemma_labels': aligned_lemma_labels
        }

    def prepare_dataset(self, sentences, ner_labels, pos_labels, lemma_labels):
        """
        Prepare dataset for multi-task BERT training
        """
        print(f"🔄 Preparing Multi-Task BERT dataset...")
        print(f"   • Total sentences: {len(sentences)}")
        print(f"   • Max length: {self.max_length}")

        all_input_ids = []
        all_attention_masks = []
        all_ner_labels = []
        all_pos_labels = []
        all_lemma_labels = []
        skipped_sentences = 0

        for i, (words, sentence_ner_labels, sentence_pos_labels, sentence_lemma_labels) in enumerate(
            zip(sentences, ner_labels, pos_labels, lemma_labels)
        ):
            if len(words) == 0:
                skipped_sentences += 1
                continue

            if not (len(words) == len(sentence_ner_labels) == len(sentence_pos_labels) == len(sentence_lemma_labels)):
                print(f"   Warning: Sentence {i} has mismatched lengths")
                continue

            # Align all labels with BERT tokens
            aligned_data = self.align_labels_with_tokens(
                words, sentence_ner_labels, sentence_pos_labels, sentence_lemma_labels
            )

            all_input_ids.append(aligned_data['input_ids'])
            all_attention_masks.append(aligned_data['attention_mask'])
            all_ner_labels.append(aligned_data['ner_labels'])
            all_pos_labels.append(aligned_data['pos_labels'])
            all_lemma_labels.append(aligned_data['lemma_labels'])

            # Progress update
            if (i + 1) % 1000 == 0:
                print(f"   Processed {i + 1}/{len(sentences)} sentences")

        print(f"   ✅ Processed {len(all_input_ids)} sentences")
        if skipped_sentences > 0:
            print(f"   ⚠️ Skipped {skipped_sentences} invalid sentences")

        # Create HuggingFace Dataset
        dataset = Dataset.from_dict({
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'ner_labels': all_ner_labels,
            'pos_labels': all_pos_labels,
            'lemma_labels': all_lemma_labels
        })

        return dataset

    def prepare_data_splits(self, test_size=0.1, val_size=0.1):
        """
        Prepare train/validation/test splits for multi-task learning
        """
        print(f"📊 Preparing multi-task data splits...")

        # Convert sentences to words and all label indices
        words = []
        ner_tag_indices = []
        pos_tag_indices = []
        lemma_tag_indices = []

        for sentence in self.sentences:
            sentence_words = [item[0] for item in sentence]  # words
            sentence_ner = [self.ner_label2id[item[1]] for item in sentence]  # NER tags
            sentence_pos = [self.pos_label2id[item[2]] for item in sentence]  # POS tags
            sentence_lemma = [self.lemma_label2id[item[3]] for item in sentence]  # Lemmas

            words.append(sentence_words)
            ner_tag_indices.append(sentence_ner)
            pos_tag_indices.append(sentence_pos)
            lemma_tag_indices.append(sentence_lemma)

        # First split: separate test set
        (words_temp, words_test,
         ner_temp, ner_test,
         pos_temp, pos_test,
         lemma_temp, lemma_test) = train_test_split(
            words, ner_tag_indices, pos_tag_indices, lemma_tag_indices,
            test_size=test_size, random_state=42
        )

        # Second split: separate train and validation
        val_size_adjusted = val_size / (1 - test_size)
        (words_train, words_val,
         ner_train, ner_val,
         pos_train, pos_val,
         lemma_train, lemma_val) = train_test_split(
            words_temp, ner_temp, pos_temp, lemma_temp,
            test_size=val_size_adjusted, random_state=42
        )

        print(f"   • Train samples: {len(words_train)}")
        print(f"   • Validation samples: {len(words_val)}")
        print(f"   • Test samples: {len(words_test)}")

        # Create datasets
        self.train_dataset = self.prepare_dataset(words_train, ner_train, pos_train, lemma_train)
        self.val_dataset = self.prepare_dataset(words_val, ner_val, pos_val, lemma_val)
        self.test_dataset = self.prepare_dataset(words_test, ner_test, pos_test, lemma_test)

        print(f"✅ Multi-task data splits prepared!")

        return self.train_dataset, self.val_dataset, self.test_dataset

    def compute_metrics(self, eval_pred):
        predictions = eval_pred.predictions  # should be [batch, seq, 3]
        labels = eval_pred.label_ids
        # Unpack tuple of predictions and labels
        ner_predictions, pos_predictions, lemma_predictions = eval_pred.predictions
        ner_labels, pos_labels, lemma_labels = eval_pred.label_ids

        ner_predictions = predictions[:, :, 0]
        pos_predictions = predictions[:, :, 1]
        lemma_predictions = predictions[:, :, 2]

        ner_labels = labels[:, :, 0]
        pos_labels = labels[:, :, 1]
        lemma_labels = labels[:, :, 2]

        # Then compute metrics for each task as before
        def compute_task_metrics(predictions, labels, task_name):
            true_predictions = [
                [p for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]
            true_labels = [
                [l for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]

            flat_true_labels = [label for sublist in true_labels for label in sublist]
            flat_predictions = [pred for sublist in true_predictions for pred in sublist]

            if len(flat_true_labels) == 0:
                return {}

            accuracy = accuracy_score(flat_true_labels, flat_predictions)
            precision, recall, f1, _ = precision_recall_fscore_support(
                flat_true_labels, flat_predictions, average='weighted', zero_division=0
            )

            return {
                f"{task_name}_accuracy": float(accuracy),
                f"{task_name}_f1": float(f1),
                f"{task_name}_precision": float(precision),
                f"{task_name}_recall": float(recall),
            }

        ner_metrics = compute_task_metrics(ner_predictions, ner_labels, "ner")
        pos_metrics = compute_task_metrics(pos_predictions, pos_labels, "pos")
        lemma_metrics = compute_task_metrics(lemma_predictions, lemma_labels, "lemma")

        all_metrics = {**ner_metrics, **pos_metrics, **lemma_metrics}
        f1_scores = [all_metrics.get(f"{task}_f1", 0) for task in ["ner", "pos", "lemma"]]
        all_metrics["average_f1"] = float(np.mean(f1_scores))

        return all_metrics


    def setup_trainer(self, output_dir='./multitask_bert_results', num_epochs=3,
                 train_batch_size=8, eval_batch_size=16):
        """
        Setup HuggingFace Trainer for multi-task learning
        """
        if self.train_dataset is None:
            print("❌ No training dataset found. Run prepare_data_splits() first.")
            return None

        # Disable wandb logging
        os.environ["WANDB_DISABLED"] = "true"

        training_args = TrainingArguments(
                output_dir=output_dir,
                num_train_epochs=num_epochs,
                per_device_train_batch_size=train_batch_size,
                per_device_eval_batch_size=eval_batch_size,
                warmup_steps=500,
                weight_decay=0.01,
                logging_dir='./multitask_bert_logs',
                logging_steps=100,
                eval_strategy="steps",  # Re-enable evaluation
                eval_steps=200,  # Evaluate every 500 steps
                save_steps=400,
                save_total_limit=2,
                load_best_model_at_end=True,
                metric_for_best_model="average_f1",  # Use average F1 across tasks
                greater_is_better=True,
                report_to=[],
                seed=42,
                fp16=torch.cuda.is_available(),
                dataloader_num_workers=2,
                remove_unused_columns=False,
                push_to_hub=False,
                optim="adamw_torch",
            )

        # Create custom trainer
        self.trainer = MultiTaskTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            data_collator=self.data_collator,
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics,
        )

        print(f"🎯 Multi-Task Trainer setup complete!")
        print(f"   • Epochs: {num_epochs}")
        print(f"   • Train batch size: {train_batch_size}")
        print(f"   • Eval batch size: {eval_batch_size}")
        print(f"   • Evaluation: Manual (after training)")

        return self.trainer

    def train(self, save_model_path='./best_multitask_bert_model'):
        """Train the multi-task BERT model"""
        if self.trainer is None:
            print("❌ Trainer not setup. Run setup_trainer() first.")
            return None

        print("🚀 STARTING MULTI-TASK BERT TRAINING")
        print("=" * 60)
        print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)

        start_time = time.time()

        try:
            # Train the model
            train_result = self.trainer.train()

            # Training completed
            end_time = time.time()
            training_time = end_time - start_time

            print("\n" + "=" * 60)
            print("🎉 MULTI-TASK BERT TRAINING COMPLETED!")
            print("=" * 60)
            print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"Total training time: {training_time/60:.1f} minutes")
            print(f"Final train loss: {train_result.training_loss:.4f}")

            # Save the model
            self.trainer.save_model(save_model_path)
            self.tokenizer.save_pretrained(save_model_path)

            # Save label mappings
            label_mappings = {
                'ner_label2id': self.ner_label2id,
                'ner_id2label': self.ner_id2label,
                'pos_label2id': self.pos_label2id,
                'pos_id2label': self.pos_id2label,
                'lemma_label2id': self.lemma_label2id,
                'lemma_id2label': self.lemma_id2label,
            }

            import json
            with open(f"{save_model_path}/label_mappings.json", 'w') as f:
                json.dump(label_mappings, f, indent=2)

            print(f"✅ Model and mappings saved to '{save_model_path}'")

            return train_result

        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def evaluate(self, dataset=None):
        """Evaluate the multi-task model"""
        if self.trainer is None:
            print("❌ Trainer not available.")
            return None

        if dataset is None:
            dataset = self.test_dataset

        if dataset is None:
            print("❌ No dataset provided and no test dataset available.")
            return None

        print("\n🔍 EVALUATING MULTI-TASK BERT MODEL")
        print("=" * 50)

        try:
            eval_result = self.trainer.evaluate(dataset)

            print("📊 Multi-Task Evaluation Results:")
            print(f"\n🎯 NER Performance:")
            print(f"   • Accuracy: {eval_result.get('eval_ner_accuracy', 0):.4f}")
            print(f"   • F1 Score: {eval_result.get('eval_ner_f1', 0):.4f}")
            print(f"   • Precision: {eval_result.get('eval_ner_precision', 0):.4f}")
            print(f"   • Recall: {eval_result.get('eval_ner_recall', 0):.4f}")

            print(f"\n🏷️ POS Performance:")
            print(f"   • Accuracy: {eval_result.get('eval_pos_accuracy', 0):.4f}")
            print(f"   • F1 Score: {eval_result.get('eval_pos_f1', 0):.4f}")
            print(f"   • Precision: {eval_result.get('eval_pos_precision', 0):.4f}")
            print(f"   • Recall: {eval_result.get('eval_pos_recall', 0):.4f}")

            print(f"\n📝 Lemma Performance:")
            print(f"   • Accuracy: {eval_result.get('eval_lemma_accuracy', 0):.4f}")
            print(f"   • F1 Score: {eval_result.get('eval_lemma_f1', 0):.4f}")
            print(f"   • Precision: {eval_result.get('eval_lemma_precision', 0):.4f}")
            print(f"   • Recall: {eval_result.get('eval_lemma_recall', 0):.4f}")

            print(f"\n🎯 Overall Performance:")
            print(f"   • Average F1: {eval_result.get('eval_average_f1', 0):.4f}")
            print(f"   • Total Loss: {eval_result.get('eval_loss', 0):.4f}")

            return eval_result

        except Exception as e:
            print(f"❌ Evaluation failed: {e}")
            return None

    def predict(self, text):
        """
        Predict NER, POS, and Lemma tags for input text

        Args:
            text: input text string or list of words

        Returns:
            list of (word, ner_tag, pos_tag, lemma) tuples
        """
        self.model.eval()

        # Handle both string and list inputs
        if isinstance(text, str):
            words = text.split()
        else:
            words = text

        # Tokenize
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Move to device
        input_data = {}
        for k, v in encoding.items():
            if k not in ['offset_mapping', 'token_type_ids']:  # Exclude problematic keys
                input_data[k] = v.to(self.device)

        # Predict
        with torch.no_grad():
            outputs = self.model(**input_data)
            ner_predictions = torch.argmax(outputs['ner_logits'], dim=2)
            pos_predictions = torch.argmax(outputs['pos_logits'], dim=2)
            lemma_predictions = torch.argmax(outputs['lemma_logits'], dim=2)

        # Align predictions with original words
        word_ids = encoding.word_ids()
        predicted_ner_tags = []
        predicted_pos_tags = []
        predicted_lemma_tags = []

        for i, word_id in enumerate(word_ids):
            if word_id is not None and i < len(ner_predictions[0]):
                ner_tag_id = ner_predictions[0][i].item()
                pos_tag_id = pos_predictions[0][i].item()
                lemma_tag_id = lemma_predictions[0][i].item()

                ner_tag = self.ner_id2label[ner_tag_id]
                pos_tag = self.pos_id2label[pos_tag_id]
                lemma_tag = self.lemma_id2label[lemma_tag_id]

                # Extend lists if needed
                while len(predicted_ner_tags) <= word_id:
                    predicted_ner_tags.append(None)
                    predicted_pos_tags.append(None)
                    predicted_lemma_tags.append(None)

                # Only assign to first subword
                if predicted_ner_tags[word_id] is None:
                    predicted_ner_tags[word_id] = ner_tag
                    predicted_pos_tags[word_id] = pos_tag
                    predicted_lemma_tags[word_id] = lemma_tag

        # Create result tuples
        result = []
        for i, word in enumerate(words):
            ner_tag = predicted_ner_tags[i] if i < len(predicted_ner_tags) and predicted_ner_tags[i] else 'O'
            pos_tag = predicted_pos_tags[i] if i < len(predicted_pos_tags) and predicted_pos_tags[i] else 'UNKNOWN'
            lemma = predicted_lemma_tags[i] if i < len(predicted_lemma_tags) and predicted_lemma_tags[i] else word.lower()

            result.append((word, ner_tag, pos_tag, lemma))

        return result

    def analyze_sentence(self, text):
        """
        Comprehensive analysis of a sentence with formatted output

        Args:
            text: input sentence string

        Returns:
            formatted analysis results
        """
        predictions = self.predict(text)

        print(f"\n🔍 SENTENCE ANALYSIS")
        print("=" * 60)
        print(f"Input: {text}")
        print("=" * 60)
        print(f"{'Word':<15} {'NER':<10} {'POS':<10} {'Lemma':<15}")
        print("-" * 60)

        for word, ner, pos, lemma in predictions:
            print(f"{word:<15} {ner:<10} {pos:<10} {lemma:<15}")

        print("=" * 60)
        return predictions

    def get_model_info(self):
        """Get information about the model"""
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)

        info = {
            'model_name': self.model_name,
            'num_labels': self.num_labels,
            'device': str(self.device),
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'model_size_mb': total_params * 4 / 1024 / 1024,
            'labels': list(self.label2id.keys())
        }

        return info

    def print_model_info(self):
        """Print detailed model information"""
        info = self.get_model_info()

        print(f"\n📊 BERT Model Information:")
        print(f"   • Model: {info['model_name']}")
        print(f"   • Number of labels: {info['num_labels']}")
        print(f"   • Device: {info['device']}")
        print(f"   • Total parameters: {info['total_parameters']:,}")
        print(f"   • Trainable parameters: {info['trainable_parameters']:,}")
        print(f"   • Model size: ~{info['model_size_mb']:.1f} MB")
        print(f"   • Labels: {info['labels']}")

In [ ]:
# Initialize the multi-task model
multitask_bert = MultiTaskBERT(dataset)

🤖 Multi-Task BERT Model Initialized:
   • Model: bert-base-multilingual-cased
   • Device: cuda
   • NER labels: 21 (['B-DATE_0', 'B-DATE_1', 'B-EVENT', 'B-ORG', 'B-PER', 'B-PRO', 'B-RRUGE', 'B-SHESH', 'B-VEND_0', 'B-VEND_1', 'I-DATE_0', 'I-DATE_1', 'I-EVENT', 'I-ORG', 'I-PER', 'I-PRO', 'I-RRUGE', 'I-SHESH', 'I-VEND_0', 'I-VEND_1', 'O'])
   • POS labels: 18 (['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_'])
   • Lemma labels: 38386


In [ ]:
# Prepare data (80% train, 10% val, 10% test)
multitask_bert.prepare_data_splits()

📊 Preparing multi-task data splits...
   • Train samples: 31380
   • Validation samples: 3923
   • Test samples: 3923
🔄 Preparing Multi-Task BERT dataset...
   • Total sentences: 31380
   • Max length: 128
   Processed 1000/31380 sentences
   Processed 2000/31380 sentences
   Processed 3000/31380 sentences
   Processed 4000/31380 sentences
   Processed 5000/31380 sentences
   Processed 6000/31380 sentences
   Processed 7000/31380 sentences
   Processed 8000/31380 sentences
   Processed 9000/31380 sentences
   Processed 10000/31380 sentences
   Processed 11000/31380 sentences
   Processed 12000/31380 sentences
   Processed 13000/31380 sentences
   Processed 14000/31380 sentences
   Processed 15000/31380 sentences
   Processed 16000/31380 sentences
   Processed 17000/31380 sentences
   Processed 18000/31380 sentences
   Processed 19000/31380 sentences
   Processed 20000/31380 sentences
   Processed 21000/31380 sentences
   Processed 22000/31380 sentences
   Processed 23000/31380 sentence

(Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels', 'lemma_labels'],
     num_rows: 31380
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels', 'lemma_labels'],
     num_rows: 3923
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels', 'lemma_labels'],
     num_rows: 3923
 }))

In [ ]:
# Setup trainer
multitask_bert.setup_trainer(num_epochs=1, train_batch_size=16)

🎯 Multi-Task Trainer setup complete!
   • Epochs: 1
   • Train batch size: 16
   • Eval batch size: 16
   • Evaluation: Manual (after training)


/tmp/ipython-input-3356205445.py:572: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MultiTaskTrainer.__init__`. Use `processing_class` instead.
  self.trainer = MultiTaskTrainer(


In [ ]:
# Train the model
train_results = multitask_bert.train()

Exception ignored in: <function _ConnectionBase.__del__ at 0x7feed02c2de0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


🚀 STARTING MULTI-TASK BERT TRAINING
Start time: 2025-09-11 21:51:48


Step,Training Loss,Validation Loss


❌ Training failed: tuple indices must be integers or slices, not tuple


In [ ]:
# Evaluate performance
eval_results = multitask_bert.evaluate()


🔍 EVALUATING MULTI-TASK BERT MODEL


📊 Multi-Task Evaluation Results:

🎯 NER Performance:
❌ Evaluation failed: 'EvalLoopOutput' object has no attribute 'get'


In [ ]:

# Analyze a sentence with formatted output
multitask_bert.analyze_sentence("Shqipëria do të marrë pjesë në Samitin e BE-së në Bruksel të hënën në 20 Maj 2020.")

TypeError: MultiTaskBERTModel.forward() got an unexpected keyword argument 'token_type_ids'

In [ ]:
class MultiTaskTrainer(Trainer):
    """Custom Trainer for multi-task learning with proper evaluation"""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Custom loss computation for multi-task learning
        """
        outputs = model(**inputs)
        loss = outputs['loss']

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """
        Custom prediction step that returns proper format for evaluation
        """
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs['loss']

            if prediction_loss_only:
                return (loss.detach(), None, None)

            # Return logits as a single concatenated tensor to avoid padding issues
            # We'll separate them later in compute_metrics
            batch_size, seq_len = outputs['ner_logits'].shape[:2]

            # Create a combined prediction tensor with task indicators
            # Format: [batch_size, seq_len, total_features]
            # where total_features = ner_logits + pos_logits + lemma_logits + task_labels
            ner_logits = outputs['ner_logits']  # [batch, seq, ner_labels]
            pos_logits = outputs['pos_logits']  # [batch, seq, pos_labels]

            # Get predictions (argmax)
            ner_preds = torch.argmax(ner_logits, dim=-1)  # [batch, seq]
            pos_preds = torch.argmax(pos_logits, dim=-1)  # [batch, seq]

        return (loss.detach(),
                (ner_preds, pos_preds),
                (inputs['ner_labels'], inputs['pos_labels']))


class MultiTaskBERTModel(nn.Module):
    """Custom BERT model with multiple classification heads for NER, POS, and Lemma"""

    def __init__(self, model_name, num_ner_labels, num_pos_labels):  # Remove num_lemma_labels
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)

        # Only two classification heads now
        self.ner_classifier = nn.Linear(self.bert.config.hidden_size, num_ner_labels)
        self.pos_classifier = nn.Linear(self.bert.config.hidden_size, num_pos_labels)

        # Simplified loss weights for two tasks
        self.ner_weight = 1.0
        self.pos_weight = 1.0

    def forward(self, input_ids, attention_mask=None, ner_labels=None, pos_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        # Only compute NER and POS logits
        ner_logits = self.ner_classifier(sequence_output)
        pos_logits = self.pos_classifier(sequence_output)

        total_loss = None
        if ner_labels is not None and pos_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            # Calculate losses for two tasks only
            ner_loss = loss_fct(ner_logits.view(-1, self.ner_classifier.out_features), ner_labels.view(-1))
            pos_loss = loss_fct(pos_logits.view(-1, self.pos_classifier.out_features), pos_labels.view(-1))

            # Combined weighted loss for two tasks
            total_loss = (self.ner_weight * ner_loss + self.pos_weight * pos_loss)

        return {
            'loss': total_loss,
            'ner_logits': ner_logits,
            'pos_logits': pos_logits,
            'ner_loss': ner_loss if ner_labels is not None else None,
            'pos_loss': pos_loss if pos_labels is not None else None,
        }


class MultiTaskBERT:
    def __init__(self, dataset, model_name="bert-base-multilingual-cased"):
        """
        Initialize Multi-Task BERT model for NER, POS tagging, and Lemmatization

        Args:
            dataset: pandas DataFrame with columns ['SENTENCE #', 'WORD', 'NER_TAG', 'POS_TAG', 'LEMMA']
            model_name: pretrained BERT model name
        """
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dataset = dataset
        self.max_length = 128

        # Get labels for all tasks
        self.ner_label2id, self.ner_id2label = self.get_labels('NER_TAG')
        self.pos_label2id, self.pos_id2label = self.get_labels('POS_TAG')

        self.num_ner_labels = len(self.ner_label2id)
        self.num_pos_labels = len(self.pos_label2id)

        self.sentences = self.get_sentences()

        # Initialize tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        # Initialize custom multi-task model
        self.model = MultiTaskBERTModel(
            model_name=self.model_name,
            num_ner_labels=self.num_ner_labels,
            num_pos_labels=self.num_pos_labels,
        ).to(self.device)

        # Custom data collator for multi-task learning
        self.data_collator = self.create_multi_task_data_collator()

        # Datasets will be created when needed
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.trainer = None

        print(f"🤖 Multi-Task BERT Model Initialized:")
        print(f"   • Model: {self.model_name}")
        print(f"   • Device: {self.device}")
        print(f"   • NER labels: {self.num_ner_labels} ({list(self.ner_label2id.keys())})")
        print(f"   • POS labels: {self.num_pos_labels} ({list(self.pos_label2id.keys())})")

    def get_labels(self, column_name):
        """Extract unique labels and create mapping dictionaries"""

        labels = sorted(list(set(self.dataset[column_name].values)))
        label2id = {label: i for i, label in enumerate(labels)}
        id2label = {v: k for k, v in label2id.items()}
        return label2id, id2label

    def get_sentences(self):
        """Convert dataset to list of (word, ner_tag, pos_tag, lemma) tuples grouped by sentence"""
        def to_tuples(group):
            return list(zip(
                group["WORD"].values,
                group["NER_TAG"].values,
                group["POS_TAG"].values
            ))

        sentences = self.dataset.groupby("SENTENCE #").apply(
            to_tuples, include_groups=False
        ).tolist()
        return sentences

    def create_multi_task_data_collator(self):
        """Create custom data collator for multi-task learning"""
        def collate_fn(features):
            batch = {}

            # Standard tokenizer collation
            input_ids = [f['input_ids'] for f in features]
            attention_mask = [f['attention_mask'] for f in features]
            ner_labels = [f['ner_labels'] for f in features]
            pos_labels = [f['pos_labels'] for f in features]

            # Pad sequences
            max_len = max(len(ids) for ids in input_ids)

            batch['input_ids'] = torch.tensor([
                ids + [self.tokenizer.pad_token_id] * (max_len - len(ids))
                for ids in input_ids
            ])

            batch['attention_mask'] = torch.tensor([
                mask + [0] * (max_len - len(mask))
                for mask in attention_mask
            ])

            batch['ner_labels'] = torch.tensor([
                labels + [-100] * (max_len - len(labels))
                for labels in ner_labels
            ])

            batch['pos_labels'] = torch.tensor([
                labels + [-100] * (max_len - len(labels))
                for labels in pos_labels
            ])

            return batch

        return collate_fn

    def align_labels_with_tokens(self, words, ner_labels, pos_labels):
        """
        Align all labels with BERT subword tokens

        Args:
            words: list of words
            ner_labels: list of NER labels (as indices)
            pos_labels: list of POS labels (as indices)
            lemma_labels: list of Lemma labels (as indices)

        Returns:
            dict with input_ids, attention_mask, and all aligned labels
        """
        tokenized_inputs = self.tokenizer(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            is_split_into_words=True,
            return_offsets_mapping=True
        )

        aligned_ner_labels = []
        aligned_pos_labels = []

        word_ids = tokenized_inputs.word_ids()
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], [PAD])
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the original labels
                aligned_ner_labels.append(ner_labels[word_idx])
                aligned_pos_labels.append(pos_labels[word_idx])
            else:
                # Subsequent subwords of the same word get -100 (ignored in loss)
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)

            previous_word_idx = word_idx

        return {
            'input_ids': tokenized_inputs['input_ids'],
            'attention_mask': tokenized_inputs['attention_mask'],
            'ner_labels': aligned_ner_labels,
            'pos_labels': aligned_pos_labels,
        }

    def prepare_dataset(self, sentences, ner_labels, pos_labels):
        """
        Prepare dataset for multi-task BERT training
        """
        print(f"🔄 Preparing Multi-Task BERT dataset...")
        print(f"   • Total sentences: {len(sentences)}")
        print(f"   • Max length: {self.max_length}")

        all_input_ids = []
        all_attention_masks = []
        all_ner_labels = []
        all_pos_labels = []
        skipped_sentences = 0

        for i, (words, sentence_ner_labels, sentence_pos_labels) in enumerate(
            zip(sentences, ner_labels, pos_labels)
        ):
            if len(words) == 0:
                skipped_sentences += 1
                continue

            if not (len(words) == len(sentence_ner_labels) == len(sentence_pos_labels)):
                print(f"   Warning: Sentence {i} has mismatched lengths")
                continue

            # Align all labels with BERT tokens
            aligned_data = self.align_labels_with_tokens(
                words, sentence_ner_labels, sentence_pos_labels
            )

            all_input_ids.append(aligned_data['input_ids'])
            all_attention_masks.append(aligned_data['attention_mask'])
            all_ner_labels.append(aligned_data['ner_labels'])
            all_pos_labels.append(aligned_data['pos_labels'])

            # Progress update
            if (i + 1) % 1000 == 0:
                print(f"   Processed {i + 1}/{len(sentences)} sentences")

        print(f"   ✅ Processed {len(all_input_ids)} sentences")
        if skipped_sentences > 0:
            print(f"   ⚠️ Skipped {skipped_sentences} invalid sentences")

        # Create HuggingFace Dataset
        dataset = Dataset.from_dict({
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'ner_labels': all_ner_labels,
            'pos_labels': all_pos_labels,
        })

        return dataset

    def prepare_data_splits(self, test_size=0.1, val_size=0.1):
        """
        Prepare train/validation/test splits for multi-task learning
        """
        print(f"📊 Preparing multi-task data splits...")

        # Convert sentences to words and all label indices
        words = []
        ner_tag_indices = []
        pos_tag_indices = []

        for sentence in self.sentences:
            sentence_words = [item[0] for item in sentence]  # words
            sentence_ner = [self.ner_label2id[item[1]] for item in sentence]  # NER tags
            sentence_pos = [self.pos_label2id[item[2]] for item in sentence]  # POS tags

            words.append(sentence_words)
            ner_tag_indices.append(sentence_ner)
            pos_tag_indices.append(sentence_pos)

        # First split: separate test set
        (words_temp, words_test,
         ner_temp, ner_test,
         pos_temp, pos_test
         ) = train_test_split(
            words, ner_tag_indices, pos_tag_indices,
            test_size=test_size, random_state=42
        )

        # Second split: separate train and validation
        val_size_adjusted = val_size / (1 - test_size)
        (words_train, words_val,
         ner_train, ner_val,
         pos_train, pos_val) = train_test_split(
            words_temp, ner_temp, pos_temp,
            test_size=val_size_adjusted, random_state=42
        )

        print(f"   • Train samples: {len(words_train)}")
        print(f"   • Validation samples: {len(words_val)}")
        print(f"   • Test samples: {len(words_test)}")

        # Create datasets
        self.train_dataset = self.prepare_dataset(words_train, ner_train, pos_train)
        self.val_dataset = self.prepare_dataset(words_val, ner_val, pos_val)
        self.test_dataset = self.prepare_dataset(words_test, ner_test, pos_test)

        print(f"✅ Multi-task data splits prepared!")

        return self.train_dataset, self.val_dataset, self.test_dataset

    def compute_metrics(self, eval_pred):
        predictions = eval_pred.predictions  # should be [batch, seq, 3]
        labels = eval_pred.label_ids


        ner_predictions, pos_predictions = predictions
        ner_labels, pos_labels = labels

        # Then compute metrics for each task as before
        def compute_task_metrics(predictions, labels, task_name):
            true_predictions = [
                [p for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]
            true_labels = [
                [l for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]

            flat_true_labels = [label for sublist in true_labels for label in sublist]
            flat_predictions = [pred for sublist in true_predictions for pred in sublist]

            if len(flat_true_labels) == 0:
                return {}

            accuracy = accuracy_score(flat_true_labels, flat_predictions)
            precision, recall, f1, _ = precision_recall_fscore_support(
                flat_true_labels, flat_predictions, average='weighted', zero_division=0
            )

            return {
                f"{task_name}_accuracy": float(accuracy),
                f"{task_name}_f1": float(f1),
                f"{task_name}_precision": float(precision),
                f"{task_name}_recall": float(recall),
            }

        ner_metrics = compute_task_metrics(ner_predictions, ner_labels, "ner")
        pos_metrics = compute_task_metrics(pos_predictions, pos_labels, "pos")

        all_metrics = {**ner_metrics, **pos_metrics}
        f1_scores = [all_metrics.get(f"{task}_f1", 0) for task in ["ner", "pos"]]
        all_metrics["average_f1"] = float(np.mean(f1_scores))

        return all_metrics


    def setup_trainer(self, output_dir='./multitask_bert_results', num_epochs=3,
                 train_batch_size=8, eval_batch_size=16):
        """
        Setup HuggingFace Trainer for multi-task learning
        """
        if self.train_dataset is None:
            print("❌ No training dataset found. Run prepare_data_splits() first.")
            return None

        # Disable wandb logging
        os.environ["WANDB_DISABLED"] = "true"

        training_args = TrainingArguments(
                output_dir=output_dir,
                num_train_epochs=num_epochs,
                per_device_train_batch_size=train_batch_size,
                per_device_eval_batch_size=eval_batch_size,
                warmup_steps=500,
                weight_decay=0.01,
                logging_dir='./multitask_bert_logs',
                logging_steps=100,
                eval_strategy="steps",  # Re-enable evaluation
                eval_steps=200,  # Evaluate every 500 steps
                save_steps=400,
                save_total_limit=2,
                load_best_model_at_end=True,
                metric_for_best_model="average_f1",  # Use average F1 across tasks
                greater_is_better=True,
                report_to=[],
                seed=42,
                fp16=torch.cuda.is_available(),
                dataloader_num_workers=2,
                remove_unused_columns=False,
                push_to_hub=False,
                optim="adamw_torch",
            )

        # Create custom trainer
        self.trainer = MultiTaskTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            data_collator=self.data_collator,
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics,
        )

        print(f"🎯 Multi-Task Trainer setup complete!")
        print(f"   • Epochs: {num_epochs}")
        print(f"   • Train batch size: {train_batch_size}")
        print(f"   • Eval batch size: {eval_batch_size}")
        print(f"   • Evaluation: Manual (after training)")

        return self.trainer

    def train(self, save_model_path='./best_multitask_bert_model'):
        """Train the multi-task BERT model"""
        if self.trainer is None:
            print("❌ Trainer not setup. Run setup_trainer() first.")
            return None

        print("🚀 STARTING MULTI-TASK BERT TRAINING")
        print("=" * 60)
        print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)

        start_time = time.time()

        try:
            # Train the model
            train_result = self.trainer.train()

            # Training completed
            end_time = time.time()
            training_time = end_time - start_time

            print("\n" + "=" * 60)
            print("🎉 MULTI-TASK BERT TRAINING COMPLETED!")
            print("=" * 60)
            print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"Total training time: {training_time/60:.1f} minutes")
            print(f"Final train loss: {train_result.training_loss:.4f}")

            # Save the model
            self.trainer.save_model(save_model_path)
            self.tokenizer.save_pretrained(save_model_path)

            # Save label mappings
            label_mappings = {
                'ner_label2id': self.ner_label2id,
                'ner_id2label': self.ner_id2label,
                'pos_label2id': self.pos_label2id,
                'pos_id2label': self.pos_id2label,
            }

            import json
            with open(f"{save_model_path}/label_mappings.json", 'w') as f:
                json.dump(label_mappings, f, indent=2)

            print(f"✅ Model and mappings saved to '{save_model_path}'")

            return train_result

        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def evaluate(self, dataset=None):
        """Evaluate the multi-task model"""
        if self.trainer is None:
            print("❌ Trainer not available.")
            return None

        if dataset is None:
            dataset = self.test_dataset

        if dataset is None:
            print("❌ No dataset provided and no test dataset available.")
            return None

        print("\n🔍 EVALUATING MULTI-TASK BERT MODEL")
        print("=" * 50)

        try:
            eval_result = self.trainer.evaluate(dataset)

            print("📊 Multi-Task Evaluation Results:")
            print(f"\n🎯 NER Performance:")
            print(f"   • Accuracy: {eval_result.get('eval_ner_accuracy', 0):.4f}")
            print(f"   • F1 Score: {eval_result.get('eval_ner_f1', 0):.4f}")
            print(f"   • Precision: {eval_result.get('eval_ner_precision', 0):.4f}")
            print(f"   • Recall: {eval_result.get('eval_ner_recall', 0):.4f}")

            print(f"\n🏷️ POS Performance:")
            print(f"   • Accuracy: {eval_result.get('eval_pos_accuracy', 0):.4f}")
            print(f"   • F1 Score: {eval_result.get('eval_pos_f1', 0):.4f}")
            print(f"   • Precision: {eval_result.get('eval_pos_precision', 0):.4f}")
            print(f"   • Recall: {eval_result.get('eval_pos_recall', 0):.4f}")


            print(f"\n🎯 Overall Performance:")
            print(f"   • Average F1: {eval_result.get('eval_average_f1', 0):.4f}")
            print(f"   • Total Loss: {eval_result.get('eval_loss', 0):.4f}")

            return eval_result

        except Exception as e:
            print(f"❌ Evaluation failed: {e}")
            return None

    def predict(self, text):
        """
        Predict NER, POS, and Lemma tags for input text

        Args:
            text: input text string or list of words

        Returns:
            list of (word, ner_tag, pos_tag, lemma) tuples
        """
        self.model.eval()

        # Handle both string and list inputs
        if isinstance(text, str):
            words = text.split()
        else:
            words = text

        # Tokenize
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Move to device
        input_data = {}
        for k, v in encoding.items():
            if k not in ['offset_mapping', 'token_type_ids']:  # Exclude problematic keys
                input_data[k] = v.to(self.device)

        # Predict
        with torch.no_grad():
            outputs = self.model(**input_data)
            ner_predictions = torch.argmax(outputs['ner_logits'], dim=2)
            pos_predictions = torch.argmax(outputs['pos_logits'], dim=2)

        # Align predictions with original words
        word_ids = encoding.word_ids()
        predicted_ner_tags = []
        predicted_pos_tags = []

        for i, word_id in enumerate(word_ids):
            if word_id is not None and i < len(ner_predictions[0]):
                ner_tag_id = ner_predictions[0][i].item()
                pos_tag_id = pos_predictions[0][i].item()

                ner_tag = self.ner_id2label[ner_tag_id]
                pos_tag = self.pos_id2label[pos_tag_id]

                # Extend lists if needed
                while len(predicted_ner_tags) <= word_id:
                    predicted_ner_tags.append(None)
                    predicted_pos_tags.append(None)

                # Only assign to first subword
                if predicted_ner_tags[word_id] is None:
                    predicted_ner_tags[word_id] = ner_tag
                    predicted_pos_tags[word_id] = pos_tag

        # Create result tuples
        result = []
        for i, word in enumerate(words):
            ner_tag = predicted_ner_tags[i] if i < len(predicted_ner_tags) and predicted_ner_tags[i] else 'O'
            pos_tag = predicted_pos_tags[i] if i < len(predicted_pos_tags) and predicted_pos_tags[i] else 'UNKNOWN'

            result.append((word, ner_tag, pos_tag))

        return result

    def analyze_sentence(self, text):
        """
        Comprehensive analysis of a sentence with formatted output

        Args:
            text: input sentence string

        Returns:
            formatted analysis results
        """
        predictions = self.predict(text)

        print(f"\n🔍 SENTENCE ANALYSIS")
        print("=" * 60)
        print(f"Input: {text}")
        print("=" * 60)
        print(f"{'Word':<15} {'NER':<10} {'POS':<10} {'Lemma':<15}")
        print("-" * 60)

        for word, ner, pos in predictions:
            print(f"{word:<15} {ner:<10} {pos:<10} ")

        print("=" * 60)
        return predictions

    def get_model_info(self):
        """Get information about the model"""
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)

        info = {
            'model_name': self.model_name,
            'num_labels': self.num_labels,
            'device': str(self.device),
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'model_size_mb': total_params * 4 / 1024 / 1024,
            'labels': list(self.label2id.keys())
        }

        return info

    def print_model_info(self):
        """Print detailed model information"""
        info = self.get_model_info()

        print(f"\n📊 BERT Model Information:")
        print(f"   • Model: {info['model_name']}")
        print(f"   • Number of labels: {info['num_labels']}")
        print(f"   • Device: {info['device']}")
        print(f"   • Total parameters: {info['total_parameters']:,}")
        print(f"   • Trainable parameters: {info['trainable_parameters']:,}")
        print(f"   • Model size: ~{info['model_size_mb']:.1f} MB")
        print(f"   • Labels: {info['labels']}")

In [ ]:
# Initialize the multi-task model
multitask_bert = MultiTaskBERT(dataset)

🤖 Multi-Task BERT Model Initialized:
   • Model: bert-base-multilingual-cased
   • Device: cuda
   • NER labels: 21 (['B-DATE_0', 'B-DATE_1', 'B-EVENT', 'B-ORG', 'B-PER', 'B-PRO', 'B-RRUGE', 'B-SHESH', 'B-VEND_0', 'B-VEND_1', 'I-DATE_0', 'I-DATE_1', 'I-EVENT', 'I-ORG', 'I-PER', 'I-PRO', 'I-RRUGE', 'I-SHESH', 'I-VEND_0', 'I-VEND_1', 'O'])
   • POS labels: 18 (['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_'])


In [ ]:
# Prepare data (80% train, 10% val, 10% test)
multitask_bert.prepare_data_splits()

📊 Preparing multi-task data splits...
   • Train samples: 31380
   • Validation samples: 3923
   • Test samples: 3923
🔄 Preparing Multi-Task BERT dataset...
   • Total sentences: 31380
   • Max length: 128
   Processed 1000/31380 sentences
   Processed 2000/31380 sentences
   Processed 3000/31380 sentences
   Processed 4000/31380 sentences
   Processed 5000/31380 sentences
   Processed 6000/31380 sentences
   Processed 7000/31380 sentences
   Processed 8000/31380 sentences
   Processed 9000/31380 sentences
   Processed 10000/31380 sentences
   Processed 11000/31380 sentences
   Processed 12000/31380 sentences
   Processed 13000/31380 sentences
   Processed 14000/31380 sentences
   Processed 15000/31380 sentences
   Processed 16000/31380 sentences
   Processed 17000/31380 sentences
   Processed 18000/31380 sentences
   Processed 19000/31380 sentences
   Processed 20000/31380 sentences
   Processed 21000/31380 sentences
   Processed 22000/31380 sentences
   Processed 23000/31380 sentence

(Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels'],
     num_rows: 31380
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels'],
     num_rows: 3923
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'ner_labels', 'pos_labels'],
     num_rows: 3923
 }))

In [ ]:
# Setup trainer
multitask_bert.setup_trainer(num_epochs=1, train_batch_size=16)

🎯 Multi-Task Trainer setup complete!
   • Epochs: 1
   • Train batch size: 16
   • Eval batch size: 16
   • Evaluation: Manual (after training)


/tmp/ipython-input-730609834.py:438: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MultiTaskTrainer.__init__`. Use `processing_class` instead.
  self.trainer = MultiTaskTrainer(


In [ ]:
# Train the model
train_results = multitask_bert.train()

🚀 STARTING MULTI-TASK BERT TRAINING
Start time: 2025-09-13 10:26:08


Step,Training Loss,Validation Loss,Ner Accuracy,Ner F1,Ner Precision,Ner Recall,Pos Accuracy,Pos F1,Pos Precision,Pos Recall,Average F1
200,0.765400,0.380402,0.967393,0.959322,0.951870,0.967393,0.947635,0.946743,0.946878,0.947635,0.953033
400,0.296400,0.229291,0.978816,0.976022,0.974648,0.978816,0.963612,0.963407,0.963332,0.963612,0.969715
600,0.240400,0.196963,0.980291,0.979486,0.979065,0.980291,0.966601,0.966452,0.966611,0.966601,0.972969
800,0.199600,0.175335,0.982528,0.981583,0.981106,0.982528,0.967937,0.967870,0.968075,0.967937,0.974726
1000,0.181900,0.156228,0.983924,0.983430,0.983501,0.983924,0.971729,0.971648,0.971707,0.971729,0.977539
1200,0.156500,0.154704,0.982439,0.982555,0.983464,0.982439,0.972402,0.972273,0.972607,0.972402,0.977414
1400,0.145500,0.136535,0.984756,0.984066,0.983747,0.984756,0.974807,0.974744,0.974798,0.974807,0.979405
1600,0.130400,0.127499,0.985241,0.985475,0.986180,0.985241,0.976322,0.976307,0.976325,0.976322,0.980891
1800,0.132200,0.121834,0.986122,0.986306,0.986669,0.986122,0.977143,0.977141,0.977179,0.977143,0.981724



🎉 MULTI-TASK BERT TRAINING COMPLETED!
End time: 2025-09-13 10:31:22
Total training time: 5.2 minutes
Final train loss: 0.4000
✅ Model and mappings saved to './best_multitask_bert_model'


In [ ]:
# Evaluate performance
eval_results = multitask_bert.evaluate()


🔍 EVALUATING MULTI-TASK BERT MODEL


📊 Multi-Task Evaluation Results:

🎯 NER Performance:
   • Accuracy: 0.9832
   • F1 Score: 0.9833
   • Precision: 0.9842
   • Recall: 0.9832

🏷️ POS Performance:
   • Accuracy: 0.9771
   • F1 Score: 0.9771
   • Precision: 0.9771
   • Recall: 0.9771

🎯 Overall Performance:
   • Average F1: 0.9802
   • Total Loss: 0.1307


In [ ]:

# Analyze a sentence with formatted output
multitask_bert.analyze_sentence("Shqipëria do të marrë pjesë në Samitin e BE-së në Bruksel të hënën në 20 Maj 2020.")


🔍 SENTENCE ANALYSIS
Input: Shqipëria do të marrë pjesë në Samitin e BE-së në Bruksel të hënën në 20 Maj 2020.
Word            NER        POS        Lemma          
------------------------------------------------------------
Shqipëria       B-VEND_1   PROPN      
do              O          PART       
të              O          PART       
marrë           O          VERB       
pjesë           O          NOUN       
në              O          ADP        
Samitin         B-EVENT    PROPN      
e               O          DET        
BE-së           B-ORG      PROPN      
në              O          ADP        
Bruksel         B-VEND_0   PROPN      
të              B-DATE_1   DET        
hënën           I-DATE_1   NOUN       
në              O          ADP        
20              B-DATE_0   NUM        
Maj             I-DATE_0   NOUN       
2020.           I-DATE_0   NUM        


[('Shqipëria', 'B-VEND_1', 'PROPN'),
 ('do', 'O', 'PART'),
 ('të', 'O', 'PART'),
 ('marrë', 'O', 'VERB'),
 ('pjesë', 'O', 'NOUN'),
 ('në', 'O', 'ADP'),
 ('Samitin', 'B-EVENT', 'PROPN'),
 ('e', 'O', 'DET'),
 ('BE-së', 'B-ORG', 'PROPN'),
 ('në', 'O', 'ADP'),
 ('Bruksel', 'B-VEND_0', 'PROPN'),
 ('të', 'B-DATE_1', 'DET'),
 ('hënën', 'I-DATE_1', 'NOUN'),
 ('në', 'O', 'ADP'),
 ('20', 'B-DATE_0', 'NUM'),
 ('Maj', 'I-DATE_0', 'NOUN'),
 ('2020.', 'I-DATE_0', 'NUM')]